# Deploy VSS 3.1 (Video Search & Summarization)

This notebook deploys the NVIDIA VSS 3.1 Blueprint on GPU-equipped cloud instances.

**What it does:**
1. Validates GPU hardware and Docker prerequisites
2. Installs and configures the NGC CLI
3. Configures Docker storage for large image pulls
4. Gets the deployment code (from a local path or GitHub)
5. Detects network configuration (internal + external IPs)
6. Runs `dev-profile.sh` to deploy the selected profile
7. Verifies all services are healthy

**Supported profiles:** `base`, `search`, `alerts`, `lvs`  
**Supported hardware:** H100, L40S, RTX PRO 6000 Blackwell, DGX SPARK

---

## Prerequisites

- Linux instance with 2+ NVIDIA GPUs (H100, L40S, RTX PRO 6000 BW, or DGX SPARK)
- NVIDIA driver 550+ and CUDA 12.x installed
- Docker Engine 24+ with Docker Compose v2
- NGC API key from [ngc.nvidia.com](https://ngc.nvidia.com)
- **500GB+ disk space** for Docker images and models. Most GPU cloud instances have a small root disk (~200-250GB) plus a large ephemeral NVMe. Section 4 will auto-detect this and move Docker/containerd storage to the NVMe.

## 1. Configuration

Set your NGC API key, deployment profile, and hardware below. These variables are used by all subsequent cells.

In [ ]:
import os

# ============================================================
# REQUIRED: Set these before running anything else
# ============================================================

NGC_CLI_API_KEY = ""          # Your NGC API key — get one at https://ngc.nvidia.com

PROFILE = "base"              # Deployment profile: base, search, alerts, lvs

HARDWARE_PROFILE = "RTXPRO6000BW"  # Hardware: RTXPRO6000BW, H100, L40S, DGX-SPARK, IGX-THOR, AGX-THOR, OTHER

# ============================================================
# OPTIONAL: Override defaults if needed
# ============================================================

# Deployment source — set ONE of these:
#   DEPLOY_SOURCE_PATH: Path to the repo cloned during launchable bringup.
#                       Must contain scripts/dev-profile.sh and deployments/.
#   Set to "" to clone from GitHub instead (requires network access).
DEPLOY_SOURCE_PATH = os.path.expanduser("~/video-search-and-summarization")

GIT_BRANCH = "3.1.0"         # Git branch or tag (only used when cloning from GitHub)

ALERTS_MODE = "verification"  # Only used when PROFILE=alerts: verification or real-time

USE_REMOTE_LLM = False        # Set True to use a remote LLM endpoint instead of local
USE_REMOTE_VLM = False        # Set True to use a remote VLM endpoint instead of local

# Network overrides (auto-detected in Section 7 if left empty)
HOST_IP_OVERRIDE = ""         # Internal IP — leave empty for auto-detect
EXTERNAL_IP_OVERRIDE = ""     # External IP — leave empty for auto-detect

In [ ]:
# ---- Validate configuration ----
import os, sys

assert NGC_CLI_API_KEY, "NGC_CLI_API_KEY is required. Get one at https://ngc.nvidia.com"
assert PROFILE in ("base", "search", "alerts", "lvs"), f"Invalid PROFILE: {PROFILE}"
assert HARDWARE_PROFILE in ("H100", "L40S", "RTXPRO6000BW", "DGX-SPARK", "IGX-THOR", "AGX-THOR", "OTHER"), \
    f"Invalid HARDWARE_PROFILE: {HARDWARE_PROFILE}"

if PROFILE == "alerts":
    assert ALERTS_MODE in ("verification", "real-time"), f"Invalid ALERTS_MODE: {ALERTS_MODE}"

if HARDWARE_PROFILE in ("DGX-SPARK", "IGX-THOR", "AGX-THOR"):
    assert PROFILE in ("base", "alerts"), \
        f"{HARDWARE_PROFILE} only supports base and alerts profiles, not {PROFILE}"

if DEPLOY_SOURCE_PATH:
    assert os.path.isdir(DEPLOY_SOURCE_PATH), f"DEPLOY_SOURCE_PATH does not exist: {DEPLOY_SOURCE_PATH}"

# Export NGC key to environment for shell cells and dev-profile.sh
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

print("Configuration valid.")
print(f"  Profile:  {PROFILE}")
print(f"  Hardware: {HARDWARE_PROFILE}")
print(f"  Source:   {DEPLOY_SOURCE_PATH or f'GitHub (branch: {GIT_BRANCH})'}")
print(f"  LLM:      {'remote' if USE_REMOTE_LLM else 'local'}")
print(f"  VLM:      {'remote' if USE_REMOTE_VLM else 'local'}")
if PROFILE == "alerts":
    print(f"  Alerts:   {ALERTS_MODE}")
print(f"  NGC key:  {NGC_CLI_API_KEY[:4]}...{NGC_CLI_API_KEY[-4:]}")

## 2. Prerequisites Check

Verify that the NVIDIA driver, CUDA, Docker, and Docker Compose are installed and functional.

In [ ]:
%%bash
set -e

echo "=== NVIDIA Driver & GPU ==="
nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader
echo ""

echo "=== GPU Count ==="
GPU_COUNT=$(nvidia-smi --query-gpu=index --format=csv,noheader | wc -l)
echo "Detected $GPU_COUNT GPU(s)"
echo ""

echo "=== Docker ==="
docker --version
docker compose version
echo ""

echo "=== NVIDIA Container Toolkit ==="
if docker run --rm --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi > /dev/null 2>&1; then
    echo "NVIDIA Container Toolkit: OK"
else
    echo "WARNING: NVIDIA Container Toolkit may not be installed."
    echo "Install: https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html"
fi
echo ""

echo "=== Disk Space ==="
df -h / | tail -1 | awk '{print "Root:", $4, "available of", $2}'
echo ""
echo "Prerequisites check complete."

## 3. Install NGC CLI

The NGC CLI is required to download models during deployment. This cell installs it if not already present, then configures it with your API key.

In [ ]:
import subprocess, os, shutil

def run(cmd, **kwargs):
    """Run a shell command, raise on failure with output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

# Check if NGC CLI is already installed
ngc_path = shutil.which("ngc")
if ngc_path:
    ver = run("ngc --version 2>&1 | head -1")
    print(f"NGC CLI already installed: {ver}")
else:
    import platform
    arch = platform.machine()
    if arch in ("aarch64", "arm64"):
        filename = "ngccli_linux_arm64.zip"
    else:
        filename = "ngccli_linux.zip"

    # Use version-pinned URL (update this if a newer version is needed)
    NGC_CLI_VERSION = "4.13.0"
    url = f"https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/{NGC_CLI_VERSION}/files/{filename}"

    print(f"Installing NGC CLI {NGC_CLI_VERSION} ...")
    run(f"cd /tmp && wget -q --content-disposition '{url}' -O ngc_cli.zip")

    # Verify download is not empty
    size = os.path.getsize("/tmp/ngc_cli.zip")
    if size < 1000:
        raise RuntimeError(f"NGC CLI download failed — file is only {size} bytes. Check the version URL.")
    print(f"  Downloaded {size / 1024 / 1024:.1f} MB")

    run("cd /tmp && unzip -o ngc_cli.zip")
    # NGC bundles its own Python — copy the entire directory
    run("sudo cp -r /tmp/ngc-cli/* /usr/local/bin/")
    run("rm -rf /tmp/ngc_cli.zip /tmp/ngc-cli")

    ver = run("ngc --version 2>&1 | head -1")
    print(f"  Installed: {ver}")

# Configure NGC CLI with API key and org
print("Configuring NGC CLI...")
ngc_dir = os.path.expanduser("~/.ngc")
os.makedirs(ngc_dir, exist_ok=True)

with open(os.path.join(ngc_dir, "config"), "w") as f:
    f.write(f""";WARNING - This is a machine generated file. Do not edit manually.
;WARNING - To update local config settings, see 'ngc config set -h'.

[CURRENT]
apikey = {NGC_CLI_API_KEY}
format_type = ascii
org = nvstaging
""")

print("NGC CLI configured.")
print(run("ngc config current"))

## 4. Docker & Containerd Storage

Docker images and containerd layers for VSS require **~250GB** (NIM models, DeepStream, ELK, etc.). Most GPU cloud instances ship with a small root disk (200-250GB) that **will run out of space** during deployment.

This cell auto-detects whether your root disk is too small and moves Docker and containerd storage to a larger mount. Docker **volumes** (Elasticsearch indices, uploaded videos, Kafka data) are kept on the root disk so your data persists even if the instance is stopped and the ephemeral NVMe is wiped. Images and layers are re-pulled automatically on next deploy.

**Common NVMe mount points** (auto-detected):
- AWS DLAMI: `/opt/dlami/nvme`
- Brev/Crusoe: `/ephemeral`
- Custom RAID: `/data`

To override auto-detection, set `STORAGE_ROOT` below.

In [ ]:
import subprocess, json, os, shutil, tempfile

STORAGE_ROOT = ""  # Override: set to a mount path (e.g. "/mnt/data") to skip auto-detection

MIN_ROOT_FREE_GB = 350  # If root has less than this free, move storage

# --- Auto-detect large mount ---

def get_disk_free_gb(path):
    """Return free space in GB for the filesystem containing path."""
    st = os.statvfs(path)
    return (st.f_bavail * st.f_frsize) / (1024 ** 3)

def get_disk_total_gb(path):
    st = os.statvfs(path)
    return (st.f_blocks * st.f_frsize) / (1024 ** 3)

def find_large_mount():
    """Look for a large non-root mount suitable for Docker storage."""
    candidates = ["/opt/dlami/nvme", "/ephemeral", "/data"]
    for path in candidates:
        if os.path.isdir(path) and os.path.ismount(path):
            free = get_disk_free_gb(path)
            if free > 200:
                return path, free
    return None, 0

def find_mount_unit(mount_path):
    """Convert a mount path to a systemd mount unit name (e.g. /opt/dlami/nvme -> opt-dlami-nvme.mount)."""
    # Strip leading slash, replace remaining slashes with dashes
    unit = mount_path.strip("/").replace("/", "-") + ".mount"
    # Verify this unit exists on the system
    r = subprocess.run(["systemctl", "cat", unit], capture_output=True, text=True)
    if r.returncode == 0:
        return unit
    return None

def get_docker_engine_version():
    """Return the Docker Engine server version, or an empty string if unavailable."""
    r = subprocess.run(["docker", "version", "--format", "{{.Server.Version}}"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return r.stdout.strip()

    print("Unable to determine Docker Engine server version; skipping Docker 29.5.x/NVCR workaround.")
    return ""

def read_daemon_config(daemon_json):
    try:
        with open(daemon_json) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def write_daemon_config(daemon_json, config):
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(config, tmp, indent=2)
        tmp.write("\n")
        tmp_path = tmp.name
    try:
        subprocess.run(["sudo", "cp", tmp_path, daemon_json], check=True)
    finally:
        os.unlink(tmp_path)

def prepare_docker_295_nvcr_workaround(config, docker_version):
    """Disable Docker's containerd image store for Docker 29.5.x NVCR pulls."""
    if not docker_version.startswith("29.5."):
        return False

    print(f"\nApplying temporary Docker 29.5.x/NVCR workaround for Docker {docker_version}.")
    print("  Setting daemon feature containerd-snapshotter=false to avoid NVCR Incorrect Repository Format pulls.")

    features = config.get("features")
    if not isinstance(features, dict):
        features = {}
        config["features"] = features

    if features.get("containerd-snapshotter") is False:
        print("  Docker 29.5.x/NVCR workaround already configured; no Docker restart needed for this patch.")
        return False

    # Temporary Docker 29.5.x/NVCR workaround: Docker 29.5.x with the
    # containerd image store can fail some nvcr.io pulls with
    # "Incorrect Repository Format". Remove this once Docker/NVCR resolves it.
    features["containerd-snapshotter"] = False
    return True

daemon_json = "/etc/docker/daemon.json"
daemon_config = read_daemon_config(daemon_json)
docker_engine_version = get_docker_engine_version()
need_docker_295_daemon_json = prepare_docker_295_nvcr_workaround(daemon_config, docker_engine_version)

root_free = get_disk_free_gb("/")
root_total = get_disk_total_gb("/")

print(f"Root disk: {root_free:.0f} GB free / {root_total:.0f} GB total")

if STORAGE_ROOT:
    large_mount = STORAGE_ROOT
    mount_free = get_disk_free_gb(STORAGE_ROOT)
    print(f"Using override: {STORAGE_ROOT} ({mount_free:.0f} GB free)")
    need_move = True
else:
    large_mount, mount_free = find_large_mount()
    need_move = root_free < MIN_ROOT_FREE_GB and large_mount is not None

    if large_mount:
        print(f"Large mount:    {large_mount} ({mount_free:.0f} GB free)")
    else:
        print("No large ephemeral mount detected.")

    if root_free >= MIN_ROOT_FREE_GB:
        print(f"\nRoot disk has enough space ({root_free:.0f} GB free). No storage move needed.")
    elif not large_mount:
        print(f"\nWARNING: Root disk only has {root_free:.0f} GB free and no large mount was found.")
        print("Deployment may fail due to disk space. Consider attaching a larger volume.")

if need_move:
    DOCKER_DATA_ROOT = os.path.join(large_mount, "docker")
    CONTAINERD_ROOT = os.path.join(large_mount, "containerd")
    VOLUMES_DIR = "/var/lib/docker/volumes"  # Keep volumes on persistent root disk

    print(f"\nMoving Docker and containerd storage to {large_mount}")
    print(f"  Docker images/layers: {DOCKER_DATA_ROOT}")
    print(f"  Containerd:           {CONTAINERD_ROOT}")
    print(f"  Docker volumes:       {VOLUMES_DIR} (stays on root for persistence)")

    # --- Check what needs changing ---
    config = daemon_config
    need_data_root_daemon_json = config.get("data-root") != DOCKER_DATA_ROOT
    need_daemon_json = need_data_root_daemon_json or need_docker_295_daemon_json

    subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT], check=True)
    subprocess.run(["sudo", "mkdir", "-p", VOLUMES_DIR], check=True)

    volumes_link = os.path.join(DOCKER_DATA_ROOT, "volumes")
    need_volumes_symlink = not (os.path.islink(volumes_link) and os.readlink(volumes_link) == VOLUMES_DIR)

    containerd_link = "/var/lib/containerd"
    need_containerd = not (os.path.islink(containerd_link) and os.readlink(containerd_link) == CONTAINERD_ROOT)

    # Even if symlinks are correct, ensure NVMe target dirs actually exist
    # (they get wiped when ephemeral NVMe is reset on instance stop/start)
    need_target_dirs = not os.path.isdir(DOCKER_DATA_ROOT) or not os.path.isdir(CONTAINERD_ROOT)
    if need_target_dirs:
        print(f"\n  NVMe target dir(s) missing — recreating...")
        subprocess.run(["sudo", "mkdir", "-p", DOCKER_DATA_ROOT, CONTAINERD_ROOT], check=True)

    if not need_daemon_json and not need_volumes_symlink and not need_containerd:
        print(f"\n  Docker data-root already set to {DOCKER_DATA_ROOT}")
        print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")
        print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

        # Always ensure the boot-time restore service is up to date
        # (handles the case where service exists but is missing mount dependencies)
        _update_restore_service = True
        _need_restart = need_target_dirs  # Restart Docker/containerd if we had to recreate dirs
    else:
        _update_restore_service = True
        _need_restart = True

        # Stop Docker AND docker.socket (socket can reactivate Docker and recreate dirs)
        print("\n  Stopping Docker and containerd for storage reconfiguration...")
        subprocess.run(["sudo", "systemctl", "stop", "docker.socket"], check=False)
        subprocess.run(["sudo", "systemctl", "stop", "docker"], check=True)
        subprocess.run(["sudo", "systemctl", "stop", "containerd"], check=True)

        # --- Docker daemon.json ---
        if need_daemon_json:
            if need_data_root_daemon_json:
                config["data-root"] = DOCKER_DATA_ROOT
            write_daemon_config(daemon_json, config)
            if need_docker_295_daemon_json:
                print("  Docker 29.5.x/NVCR workaround written to daemon.json")
            if need_data_root_daemon_json:
                print(f"  Docker data-root set to {DOCKER_DATA_ROOT}")
            else:
                print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")
        else:
            print(f"  Docker data-root already set to {DOCKER_DATA_ROOT}")

        # --- Volumes symlink (use ln -sfn for idempotency) ---
        if need_volumes_symlink:
            # ln -sfn: force, no-dereference (replaces existing dir/symlink atomically)
            subprocess.run(["sudo", "rm", "-rf", volumes_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", VOLUMES_DIR, volumes_link], check=True)
            print(f"  Created symlink: {volumes_link} -> {VOLUMES_DIR}")
        else:
            print(f"  Volumes symlink already correct: {volumes_link} -> {VOLUMES_DIR}")

        # --- Containerd ---
        if need_containerd:
            subprocess.run(["sudo", "mkdir", "-p", CONTAINERD_ROOT], check=True)
            if os.path.isdir(containerd_link) and not os.path.islink(containerd_link):
                # Move existing containerd data
                subprocess.run(f"sudo mv {containerd_link}/* {CONTAINERD_ROOT}/ 2>/dev/null; true",
                               shell=True, check=False)
                subprocess.run(["sudo", "rm", "-rf", containerd_link], check=True)
                print(f"  Containerd data moved to {CONTAINERD_ROOT}")
            elif os.path.lexists(containerd_link):
                subprocess.run(["sudo", "rm", "-f", containerd_link], check=True)
            subprocess.run(["sudo", "ln", "-sfn", CONTAINERD_ROOT, containerd_link], check=True)
            print(f"  Containerd symlinked: {containerd_link} -> {CONTAINERD_ROOT}")
        else:
            print(f"  Containerd already symlinked: {containerd_link} -> {CONTAINERD_ROOT}")

    # --- Install/update boot-time restore service ---
    # Ephemeral NVMe is wiped on instance stop/start. This systemd service
    # recreates the directories before Docker/containerd start so they don't crash-loop.
    # We use RequiresMountsFor= so the service waits for the NVMe to actually be mounted.
    if _update_restore_service:
        unit_name = "docker-nvme-restore.service"
        unit_path = f"/etc/systemd/system/{unit_name}"

        # Build After= line — include the mount unit if systemd knows about it
        after_targets = "local-fs.target"
        mount_unit = find_mount_unit(large_mount)
        if mount_unit:
            after_targets += f" {mount_unit}"

        unit_content = f"""[Unit]
Description=Restore Docker/containerd dirs on ephemeral NVMe
Before=containerd.service docker.service
After={after_targets}
RequiresMountsFor={large_mount}

[Service]
Type=oneshot
ExecStart=/bin/bash -c 'mkdir -p {DOCKER_DATA_ROOT} {CONTAINERD_ROOT}'

[Install]
WantedBy=multi-user.target
"""
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.service', delete=False) as tmp:
            tmp.write(unit_content)
            tmp_path = tmp.name
        subprocess.run(["sudo", "cp", tmp_path, unit_path], check=True)
        os.unlink(tmp_path)
        subprocess.run(["sudo", "systemctl", "daemon-reload"], check=True)
        subprocess.run(["sudo", "systemctl", "enable", unit_name], check=True, capture_output=True)
        print(f"  Installed {unit_name} (restores NVMe dirs on boot, waits for mount)")

    # --- Restart if needed ---
    if _need_restart:
        print("\n  Starting containerd and Docker...")
        subprocess.run(["sudo", "systemctl", "start", "containerd"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker.socket"], check=True)
        subprocess.run(["sudo", "systemctl", "start", "docker"], check=True)

    r = subprocess.run(["docker", "info", "--format", "{{.DockerRootDir}}"],
                       capture_output=True, text=True)
    print(f"\n  Docker data-root: {r.stdout.strip()}")
    target = os.readlink(containerd_link) if os.path.islink(containerd_link) else containerd_link
    print(f"  Containerd root:  {target}")
    print(f"\n  Storage configuration complete.")
else:
    if need_docker_295_daemon_json:
        write_daemon_config(daemon_json, daemon_config)
        print("  Docker 29.5.x/NVCR workaround written to daemon.json")
        print("\n  Restarting Docker to activate daemon.json change...")
        subprocess.run(["sudo", "systemctl", "restart", "docker"], check=True)
    if not STORAGE_ROOT and root_free >= MIN_ROOT_FREE_GB:
        print("Skipping storage move.")

## 5. Docker Login

Authenticate with the NVIDIA Container Registry (`nvcr.io`) to pull deployment images.

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "login", "nvcr.io",
     "--username", "$oauthtoken",
     "--password", NGC_CLI_API_KEY],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Docker login to nvcr.io: OK")
else:
    print(f"Docker login FAILED:\n{result.stderr}")
    raise RuntimeError("Docker login to nvcr.io failed")

## 6. Get Deployment Code

This cell locates the deployment code (scripts, compose files, configs). There are two ways to get it onto the server:

### Option 1 — Brev Launchable (automatic)

When configured as a Brev Launchable, the git repository is cloned onto the instance automatically. `DEPLOY_SOURCE_PATH` in Section 1 defaults to that launchable bringup clone path (typically `~/video-search-and-summarization`).

### Option 2 — Manual tarball

If the code isn't already on the server, create a tarball from your local checkout and copy it over:

```bash
# On your local machine:
cd /path/to/video-search-and-summarization
tar czf ~/vss-deploy-3.1.0.tar.gz --exclude='.git' .
scp ~/vss-deploy-3.1.0.tar.gz <user>@<server>:~/

# On the server:
mkdir -p ~/video-search-and-summarization
tar xzf ~/vss-deploy-3.1.0.tar.gz -C ~/video-search-and-summarization
```

Then set in **Section 1**: `DEPLOY_SOURCE_PATH = "/home/<user>/video-search-and-summarization"`

---

If `DEPLOY_SOURCE_PATH` is set, uses the code at that path directly. Set `DEPLOY_SOURCE_PATH = ""` to clone from GitHub instead (requires the repo to be accessible).

In [ ]:
import subprocess, os

if DEPLOY_SOURCE_PATH:
    # --- Use pre-extracted local repo ---
    REPO_DIR = DEPLOY_SOURCE_PATH
    print(f"Using local deployment source: {REPO_DIR}")
else:
    # --- Clone from GitHub ---
    DEPLOY_DIR = os.path.expanduser("~/deployments")
    REPO_DIR = os.path.join(DEPLOY_DIR, "video-search-and-summarization")
    GITHUB_REPO = "https://github.com/NVIDIA-AI-IOT/video-search-and-summarization.git"

    os.makedirs(DEPLOY_DIR, exist_ok=True)

    if os.path.isdir(REPO_DIR):
        print(f"Repo already exists at {REPO_DIR}")
        print(f"Fetching latest and checking out {GIT_BRANCH}...")
        subprocess.run(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR, check=True,
                       capture_output=True)
        result = subprocess.run(["git", "checkout", GIT_BRANCH], cwd=REPO_DIR,
                               capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"Failed to checkout {GIT_BRANCH}:\n{result.stderr}")
        subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=False,
                       capture_output=True)
    else:
        print(f"Cloning from GitHub (branch: {GIT_BRANCH})...")
        result = subprocess.run(
            ["git", "clone", "--branch", GIT_BRANCH, "--single-branch", GITHUB_REPO, REPO_DIR],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f"Clone failed:\n{result.stderr}")
        print("Clone complete.")

# Validate repo structure
SCRIPT_DIR = os.path.join(REPO_DIR, "scripts")
assert os.path.isfile(os.path.join(SCRIPT_DIR, "dev-profile.sh")), \
    f"dev-profile.sh not found in {SCRIPT_DIR}"
assert os.path.isdir(os.path.join(REPO_DIR, "deployments")), \
    f"deployments/ not found in {REPO_DIR}"

# Show commit info (if it's a git repo)
commit = "(not a git repo)"
branch = ""
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    commit = subprocess.run(
        ["git", "log", "--oneline", "-1"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()
    branch = subprocess.run(
        ["git", "branch", "--show-current"],
        cwd=REPO_DIR, capture_output=True, text=True
    ).stdout.strip()

print(f"\nRepo:    {REPO_DIR}")
if branch:
    print(f"Branch:  {branch}")
print(f"Commit:  {commit}")
print(f"Scripts: {SCRIPT_DIR}")
print(f"\nContents of deployments/:")
for entry in sorted(os.listdir(os.path.join(REPO_DIR, "deployments"))):
    print(f"  {entry}")

## 7. Detect Network Configuration

Auto-detects internal (`HOST_IP`) and external (`EXTERNAL_IP`) addresses. On NAT'd cloud instances (Brev, AWS), these are different — the internal IP is used for inter-container communication while the external IP is used for browser access.

If auto-detection fails or gives the wrong result, set the overrides in Section 1.

In [ ]:
import subprocess, os

def detect_internal_ip():
    """Detect internal IP via ip route (same method as dev-profile.sh)."""
    try:
        out = subprocess.run(
            ["bash", "-c", "ip route get 1.1.1.1 | awk '/src/ {for (i=1;i<=NF;i++) if ($i==\"src\") print $(i+1)}'"],
            capture_output=True, text=True, timeout=5
        )
        return out.stdout.strip()
    except Exception:
        return ""

def detect_external_ip():
    """Detect external IP via public service."""
    for cmd in ["curl -s --max-time 5 ifconfig.me", "curl -s --max-time 5 icanhazip.com"]:
        try:
            out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
            ip = out.stdout.strip()
            if ip:
                return ip
        except Exception:
            continue
    return ""

def read_etc_environment():
    """Read key=value pairs from /etc/environment (Brev sets BREV_ENV_ID there)."""
    env = {}
    try:
        with open("/etc/environment") as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    key, _, value = line.partition("=")
                    env[key.strip()] = value.strip().strip('"')
    except FileNotFoundError:
        pass
    return env

HOST_IP = HOST_IP_OVERRIDE or detect_internal_ip()
EXTERNAL_IP = EXTERNAL_IP_OVERRIDE or detect_external_ip()

print(f"Internal IP (HOST_IP):   {HOST_IP}")
print(f"External IP:             {EXTERNAL_IP}")

if HOST_IP == EXTERNAL_IP:
    print("\nInternal == External (direct connection, no NAT)")
else:
    print("\nNAT detected — internal and external IPs differ.")
    print("The deploy script will set EXTERNAL_IP automatically.")

if not HOST_IP:
    print("\nWARNING: Could not detect internal IP. Set HOST_IP_OVERRIDE in Section 1.")
if not EXTERNAL_IP:
    print("\nWARNING: Could not detect external IP. Set EXTERNAL_IP_OVERRIDE in Section 1.")

# --- Brev Secure Links ---
# On Brev, all browser-facing traffic routes through an nginx reverse proxy
# on a single port (default 7777). This avoids CORS issues with Cloudflare
# Access when each port gets its own hostname.
# Check os.environ first, then fall back to /etc/environment (Jupyter kernels
# may not inherit /etc/environment depending on how the notebook server starts).
_etc_env = read_etc_environment()
BREV_ENV_ID = os.environ.get("BREV_ENV_ID") or _etc_env.get("BREV_ENV_ID", "")
if BREV_ENV_ID:
    # Ensure it's in os.environ so dev-profile.sh picks it up
    os.environ["BREV_ENV_ID"] = BREV_ENV_ID
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    # Brev secure links use the port number as the hostname prefix
    # (e.g. port 7777 -> "7777-xxx.brevlab.com"). Set BREV_LINK_PREFIX to
    # override if your setup differs.
    brev_link_prefix = os.environ.get("BREV_LINK_PREFIX", f"{proxy_port}")
    os.environ["BREV_LINK_PREFIX"] = brev_link_prefix
    brev_ui_url = f"https://{brev_link_prefix}-{BREV_ENV_ID}.brevlab.com"
    print(f"\n=== Brev Environment Detected ===")
    print(f"  BREV_ENV_ID: {BREV_ENV_ID}")
    print(f"  Secure link prefix: {brev_link_prefix} (set BREV_LINK_PREFIX to override)")
    print(f"  All browser-facing URLs route through nginx proxy (port {proxy_port})")
    print(f"  UI will be available at: {brev_ui_url}")
else:
    BREV_ENV_ID = ""  # ensure defined for later cells

## 8. Deploy Profile

This is the main deployment cell. It runs `dev-profile.sh up` with all the configuration from Section 1 and the network settings from Section 7.

This will:
- Generate environment files
- Download required models from NGC
- Pull and build Docker images
- Start all containers

**This cell takes 10-30 minutes** depending on network speed and whether images are cached.

The cell shows a live progress summary. Full output is captured to `~/deploy_vss.log` — if something fails, check that file for details.

In [ ]:
import subprocess, os, re, time, datetime
from IPython.display import display, clear_output, HTML

LOG_FILE = os.path.expanduser("~/deploy_vss.log")

# Build the dev-profile.sh command
# NGC_CLI_API_KEY is passed via environment (no longer a CLI flag)
cmd = [
    "bash", os.path.join(SCRIPT_DIR, "dev-profile.sh"), "up",
    "--profile", PROFILE,
    "--hardware-profile", HARDWARE_PROFILE,
    "--host-ip", HOST_IP,
]

if EXTERNAL_IP and EXTERNAL_IP != HOST_IP:
    cmd += ["--external-ip", EXTERNAL_IP]

if USE_REMOTE_LLM:
    cmd += ["--use-remote-llm"]

if USE_REMOTE_VLM:
    cmd += ["--use-remote-vlm"]

if PROFILE == "alerts":
    cmd += ["--mode", ALERTS_MODE]

# Print the command
display_cmd = " ".join(cmd)
print(f"Command: {display_cmd}")
print(f"Full log: {LOG_FILE}\n")

# --- Phase detection and filtering ---

# Lines matching these patterns are noise — suppress them
SUPPRESS_PATTERNS = [
    re.compile(r"^(\s*[\u2800-\u28FF]|⠋|⠙|⠹|⠸|⠴|⠦|⠧|⠏)"),  # NGC spinner frames
    re.compile(r"^\s*M\u2026|^\s*$"),                              # Truncated NGC progress fragments
    re.compile(r"^\s*#\d+\s+sha256:"),                             # Docker buildkit layer download/extract progress
    re.compile(r"^\s*#\d+\s+extracting\s"),                        # Docker buildkit layer extraction
    re.compile(r"^\s*#\d+\s+\.\.\."),                              # Docker buildkit continuation
    re.compile(r"^time=.*level=warning"),                           # Docker compose unset variable warnings
    re.compile(r"^WARNING! Using --password"),                      # Docker login warning
    re.compile(r"Login Succeeded"),                                 # Docker login success (we print our own)
    re.compile(r"^\s*Getting files to download"),                   # NGC download preamble
    re.compile(r"^\s*━"),                                           # NGC progress bars
    re.compile(r"^\s*[0-9a-f]{12}\s+(Downloading|Extracting|Waiting|Verifying|Pull complete)"),  # Docker layer progress
]

# Lines matching these indicate phase transitions — always show
PHASE_PATTERNS = [
    (re.compile(r"\[INFO\] Generating environment"), "Generating environment"),
    (re.compile(r"\[INFO\] Downloading.*models"), "Downloading models from NGC"),
    (re.compile(r"\[INFO\].*models downloaded"), "Models downloaded"),
    (re.compile(r"\[INFO\] Logging into nvcr"), "Docker login"),
    (re.compile(r"\[INFO\] Starting docker compose"), "Starting Docker Compose"),
    (re.compile(r"\[INFO\] State up completed"), "Deployment complete"),
]

# Image pull tracking — service-level "Pulling <name>" / "Pulled <name>"
PULLING_RE = re.compile(r"^\s*Pulling\s+(\S+)")
PULLED_RE = re.compile(r"^\s*Pulled\s+(\S+)")

# Image build tracking — "#N [service-name step/total] COMMAND" / "#N DONE Ns"
BUILD_STEP_RE = re.compile(r"^\s*#\d+\s+\[(\S+)\s+(\d+/\d+)\]")
BUILD_DONE_RE = re.compile(r"^\s*#\d+\s+DONE\s+[\d.]+s")
IMAGE_BUILT_RE = re.compile(r"^\s*Image\s+(\S+)\s+Built")

# Container lifecycle — track creating/starting/healthy
CONTAINER_RE = re.compile(r"^\s*Container\s+(\S+)\s+(Creating|Created|Starting|Started|Healthy|Waiting|Exited.*)")

phases_seen = []
images_pulling = set()   # images we've seen "Pulling" for
images_pulled = set()    # images we've seen "Pulled" for
builds = {}              # service -> "step/total" for active builds
builds_done = set()      # services that finished building
containers = {}
errors = []
start_time = time.time()

def elapsed():
    s = int(time.time() - start_time)
    return f"{s // 60}m {s % 60:02d}s"

def print_status():
    clear_output(wait=True)
    print(f"Command: {display_cmd}")
    print(f"Full log: {LOG_FILE}\n")

    # Phases
    for p in phases_seen:
        print(f"  [done]  {p}")
    if phases_seen:
        print()

    # Image pull progress
    if images_pulling:
        total = len(images_pulling)
        done = len(images_pulled)
        if done < total:
            still_pulling = sorted(images_pulling - images_pulled)
            print(f"  Pulling images: {done}/{total} complete  ({elapsed()})")
            for img in still_pulling:
                print(f"    {img:<45s} pulling...")
            print()
        else:
            print(f"  Pulling images: {total}/{total} complete\n")

    # Image build progress
    active_builds = {s: step for s, step in builds.items() if s not in builds_done}
    if builds:
        done_count = len(builds_done)
        total_count = len(builds)
        if active_builds:
            print(f"  Building images: {done_count}/{total_count} complete  ({elapsed()})")
            for svc, step in sorted(active_builds.items()):
                print(f"    {svc:<45s} [{step}]")
            print()
        else:
            print(f"  Building images: {total_count}/{total_count} complete\n")

    # Container summary
    if containers:
        healthy = sum(1 for s in containers.values() if s == "Healthy")
        started = sum(1 for s in containers.values() if s in ("Started", "Healthy"))
        total = len(containers)
        print(f"  Containers: {started}/{total} started, {healthy}/{total} healthy  ({elapsed()})")

        # Show containers that aren't healthy yet
        pending = {n: s for n, s in containers.items() if s != "Healthy" and s not in ("Exited",)}
        if pending:
            # Only show non-trivial pending (skip init containers that exited)
            waiting = {n: s for n, s in pending.items() if "Exited" not in s}
            if waiting:
                print()
                for name, status in sorted(waiting.items()):
                    print(f"    {name:<45s} {status}")
        print()

    # Errors
    for e in errors:
        print(f"  ERROR: {e}")

# Run the process
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=SCRIPT_DIR,
    env={**os.environ, "NGC_CLI_API_KEY": NGC_CLI_API_KEY}
)

last_refresh = 0
with open(LOG_FILE, "w") as log:
    for line in process.stdout:
        log.write(line)
        log.flush()
        stripped = line.rstrip()

        # Capture errors
        if "[ERROR]" in stripped:
            errors.append(stripped)
            print_status()
            continue

        # Track image pulls (before suppression check)
        m_pulling = PULLING_RE.match(stripped)
        if m_pulling:
            images_pulling.add(m_pulling.group(1))
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        m_pulled = PULLED_RE.match(stripped)
        if m_pulled:
            images_pulled.add(m_pulled.group(1))
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        # Track image builds
        m_build = BUILD_STEP_RE.match(stripped)
        if m_build:
            svc, step = m_build.group(1), m_build.group(2)
            builds[svc] = step
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        m_built = IMAGE_BUILT_RE.match(stripped)
        if m_built:
            svc = m_built.group(1)
            builds_done.add(svc)
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()
            continue

        # Suppress noise
        if any(p.search(stripped) for p in SUPPRESS_PATTERNS):
            continue

        # Detect phase transitions
        for pattern, label in PHASE_PATTERNS:
            if pattern.search(stripped):
                if label not in phases_seen:
                    phases_seen.append(label)
                    print_status()
                break

        # Track container lifecycle
        m = CONTAINER_RE.match(stripped)
        if m:
            name, status = m.group(1), m.group(2)
            # Normalize "Exited (0) ..." to "Exited"
            if status.startswith("Exited"):
                status = "Exited"
            containers[name] = status
            # Refresh display at most every 2 seconds to avoid flicker
            now = time.time()
            if now - last_refresh > 2:
                last_refresh = now
                print_status()

process.wait()

# Final status
print_status()
print("=" * 50)
if process.returncode == 0 and not errors:
    print(f"Deployment complete in {elapsed()}.")
else:
    print(f"\nDeployment FAILED (exit code {process.returncode}).")
    if errors:
        print(f"\n{len(errors)} error(s) found — see above.")
    print(f"\nFull log: {LOG_FILE}")
    print(f"  View with: cat {LOG_FILE}")

## 9. Verify Deployment

Check that all containers are running and core services are healthy. The health checks poll with retries since some services take a few minutes to fully start.

In [ ]:
import subprocess, time, urllib.request, urllib.error, os

# Show running containers
print("=== Running Containers ===")
subprocess.run(["docker", "ps", "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"])
print()

# Determine proxy port
proxy_port = os.environ.get("PROXY_PORT", "7777")

# Health check endpoints by profile
checks = [
    ("Proxy",  f"http://localhost:{proxy_port}/health"),
    ("Agent",  "http://localhost:8000/health"),
    ("VST",    "http://localhost:30888/vst/api/v1/sensor/list"),
    ("UI",     "http://localhost:3000"),
]
if PROFILE in ("search", "alerts", "lvs"):
    checks += [
        ("Elasticsearch", "http://localhost:9200"),
        ("Kibana",        "http://localhost:5601/api/status"),
    ]
if PROFILE == "alerts":
    checks.append(("Video Analytics API", "http://localhost:8081/livez"))

# Poll with retries
MAX_RETRIES = 30
RETRY_INTERVAL = 10
results = {}

print(f"=== Health Checks (up to {MAX_RETRIES * RETRY_INTERVAL}s) ===")
pending = list(checks)

for attempt in range(1, MAX_RETRIES + 1):
    still_pending = []
    for name, url in pending:
        try:
            req = urllib.request.urlopen(url, timeout=5)
            results[name] = f"OK ({req.getcode()})"
        except Exception:
            still_pending.append((name, url))
    pending = still_pending
    if not pending:
        break
    waiting = ", ".join(n for n, _ in pending)
    print(f"  [{attempt}/{MAX_RETRIES}] Waiting for: {waiting}")
    time.sleep(RETRY_INTERVAL)

for name, url in pending:
    results[name] = "FAILED"

print()
all_ok = True
for name, status in results.items():
    marker = "OK" if "OK" in status else "FAIL"
    if marker == "FAIL":
        all_ok = False
    print(f"  {name:.<30s} {status}")

# Check perception container status (no HTTP health endpoint — DeepStream pipeline)
if PROFILE == "search":
    print()
    r = subprocess.run(
        ["docker", "ps", "--filter", "name=perception", "--format", "{{.Names}}: {{.Status}}"],
        capture_output=True, text=True
    )
    if r.stdout.strip():
        print("  Perception containers:")
        for line in r.stdout.strip().splitlines():
            print(f"    {line}")
    else:
        print("  WARNING: No perception containers found (required for search profile).")
        all_ok = False

print()
if all_ok:
    print("All services healthy.")
else:
    print("Some services failed to start. Check container logs:")
    print("  docker compose -p mdx logs <service-name>")

## 10. Access the UI

Once deployment is verified, open the VSS UI in your browser. Accessing the front-end will open up the chat interface where you can interact with the agent and should look like this:

![VSS UI Chat Interface](images/vss_ui_chat.png)

Run the cell below to generate your VSS UI URL.

**On Brev:** All browser-facing traffic routes through an nginx reverse proxy on a single port (default 7777). Create **one** Brev secure link for port 7777 in the dashboard — no individual port forwarding needed. For the **alerts** and **lvs** profiles, you will also need separate secure links for Kibana and other services (see table below).

**On other cloud providers:** Depending on your CSP's firewall and security group configuration, you may need to expose or forward ports to access the UI and other services from your browser. The following ports are used by VSS:

| Port | Service | Profiles | Brev Secure Link |
|------|---------|----------|------------------|
| 7777 | Nginx proxy (consolidates UI, Agent, VST) | all | Required (primary) |
| 3000 | VSS UI | all | Not needed (behind proxy) |
| 8000 | VSS Agent API | all | Not needed (behind proxy) |
| 30888 | VST (Video Storage Toolkit) | all | Not needed (behind proxy) |
| 5601 | Kibana | search, alerts, lvs | Required (separate link) |
| 6006 | Phoenix (LLM tracing/observability) | all | Optional |
| 9200 | Elasticsearch | alerts, lvs | Not needed |
| 8081 | Video Analytics API | alerts | Not needed |
| 31000 | nvstreamer (WebRTC live view) | search, alerts | Required for live camera view |
| 8554 | RTSP (if using test stream) | alerts | Not needed |

**Brev summary:** For the **base** profile, create 1 secure link (port 7777). For **search**, create secure links for ports 7777, 5601, and 31000. For **alerts** or **lvs**, create secure links for ports 7777, 5601, and 31000 (alerts only). Port 6006 (Phoenix) is optional for debugging.

If direct access is not possible, use SSH port forwarding or your CSP's port sharing/tunneling feature.

In [ ]:
import os

if BREV_ENV_ID:
    proxy_port = os.environ.get("PROXY_PORT", "7777")
    brev_link_prefix = os.environ.get("BREV_LINK_PREFIX", f"{proxy_port}")
    ui_url = f"https://{brev_link_prefix}-{BREV_ENV_ID}.brevlab.com"
    print(f"VSS UI (via Brev secure link): {ui_url}")
    print()
    print("Setup:")
    print(f"  1. Ensure a Brev secure link exists for port {proxy_port}")
    print(f"  2. Open: {ui_url}")
    print()
    print("All services (Agent API, VST, UI) are consolidated behind the proxy.")
    print("No individual port forwarding is needed.")
    if PROFILE in ("search", "alerts", "lvs"):
        print()
        print(f"=== Additional Secure Links ({PROFILE}) ===")
        print("Create these additional secure links in the Brev dashboard:")
        print()
        kibana_url = f"https://5601-{BREV_ENV_ID}.brevlab.com"
        print(f"  Kibana (port 5601):       {kibana_url}")
        if PROFILE == "alerts":
            nvstreamer_url = f"https://31000-{BREV_ENV_ID}.brevlab.com"
            print(f"  nvstreamer (port 31000):  {nvstreamer_url}")
        phoenix_url = f"https://6006-{BREV_ENV_ID}.brevlab.com"
        print(f"  Phoenix (port 6006):      {phoenix_url}  (optional, for LLM tracing)")
else:
    ui_url = f"http://{EXTERNAL_IP or HOST_IP}:3000"
    print(f"VSS UI: {ui_url}")
    print()
    print("If the URL is not directly accessible, use one of these methods:")
    print()
    print("  SSH port forwarding (works everywhere):")
    print(f"    ssh -L 3000:localhost:3000 <user>@{EXTERNAL_IP or HOST_IP}")
    print(f"    Then open: http://localhost:3000")
    print()
    print("  VSCode Remote SSH:")
    print("    Connect to the instance via Remote-SSH, ports forward automatically.")
    print()
    if PROFILE in ("search", "alerts", "lvs"):
        print(f"  Kibana dashboard: http://{EXTERNAL_IP or HOST_IP}:5601")
        if PROFILE == "alerts":
            print(f"  nvstreamer (live view): http://{EXTERNAL_IP or HOST_IP}:31000")
        print(f"  Phoenix (LLM tracing): http://{EXTERNAL_IP or HOST_IP}:6006")

## 11. Next Steps

Once you can access the VSS frontend, continue the QuickStart example which involves uploading a video, engaging in Q&A, and generating a report: [Quickstart - Upload a Video](https://docs.nvidia.com/vss/3.1.0/quickstart.html#step-2-upload-a-video)

You can either use your own videos for these examples or download the [VSS Sample Data from NGC](https://docs.nvidia.com/vss/3.1.0/quickstart.html#download-sample-data-from-ngc).

Once you've gone through the QuickStart example, you can follow **Step 12** in this notebook to deploy different [Agent Workflows](https://docs.nvidia.com/vss/3.1.0/adding-workflows.html).

**Step 13** provides instructions on stopping the deployment.

## 12. Profile-Specific Next Steps

Quick-start instructions for your deployed profile.

In [ ]:
if PROFILE == "base":
    print("""=== Base Profile — Quick Start ===

1. Open the VSS UI (see Section 10 for the URL).

2. Upload a video using the "Upload" button in the sidebar.
   Supported formats: MP4, AVI, MOV. Wait for the upload to complete.

3. Once uploaded, select the video and start chatting about it.
   Try asking: "What is happening in this video?"

4. The agent uses the VLM to analyze video frames and answers
   questions about the content.
""")

elif PROFILE == "search":
    print("""=== Search Profile — Quick Start ===

1. Open the VSS UI and switch to the "Search" tab.

2. Upload a video using the upload button. The video will be
   split into chunks and embedded for semantic search. This
   takes a few minutes depending on video length.

3. Once processing completes, use the search bar to find moments:
   - "person walking"
   - "red car"
   - "someone carrying a box"

4. Click a search result to play the matching video clip.

5. You can also chat about uploaded videos in the "Chat" tab.

Note: The perception-2d container must be running for the embedding
pipeline. Check with: docker ps | grep perception
""")

elif PROFILE == "alerts":
    print("""=== Alerts Profile — Quick Start ===

1. Open the VSS UI. The alerts profile needs an RTSP camera stream
   to generate detections and alerts.

2. Add a camera sensor:
   - Go to the "Sensors" or camera management section in the UI
   - Add your RTSP stream URL (e.g. rtsp://IP:8554/stream)
   - The perception pipeline will begin analyzing the stream

3. View live detections:
   - Open the "Alerts" tab to see real-time alerts as they're generated
   - Click an alert to view the video clip with bounding boxes

4. Open the "Dashboard" tab to see the Kibana analytics dashboard
   with detection statistics, timelines, and heatmaps.

5. Use the "Chat" tab to ask questions about detected events:
   - "What alerts happened in the last hour?"
   - "How many people were detected today?"
""")

elif PROFILE == "lvs":
    print("""=== LVS Profile — Quick Start ===

1. Open the VSS UI and upload a video via the sidebar.

2. Once uploaded, use the chat to request a report:
   - "Generate a report for my_video.mp4"
   - "Summarize what happens in this video"

3. The agent analyzes the full video and generates a structured
   report with timeline summaries, detected events, and analytics.

4. Reports are saved and accessible via the "Reports" section.
""")

## 13. Stop Deployment

Stop all containers **without deleting data or volumes**. Use this when you want to:
- Free up GPU/memory resources temporarily
- Change to a different profile (update `PROFILE` in Section 1, then re-run from Section 8)
- Restart the deployment later by re-running Section 8

In [ ]:
import subprocess, os, glob

# Find the generated.env file for the current profile
env_file = None
gen_pattern = os.path.join(REPO_DIR, "deployments", "developer-workflow", f"dev-profile-{PROFILE}", "generated.env")
matches = glob.glob(gen_pattern)
if matches:
    env_file = matches[0]

if not env_file or not os.path.isfile(env_file):
    print(f"ERROR: Could not find generated.env at {gen_pattern}")
    print("Has the deployment been run at least once (Section 8)?")
    raise FileNotFoundError(gen_pattern)

print(f"Using env file: {env_file}")
print("Stopping all VSS containers (preserving data and volumes)...\n")

result = subprocess.run(
    ["docker", "compose", "--env-file", env_file,
     "-f", os.path.join(REPO_DIR, "deployments", "compose.yml"),
     "-p", "mdx", "stop"],
    capture_output=True, text=True,
    cwd=os.path.join(REPO_DIR, "deployments")
)
print(result.stdout)
if result.stderr:
    # Filter out the harmless "variable is not set" warnings
    for line in result.stderr.splitlines():
        if "is not set" not in line:
            print(line)

if result.returncode == 0:
    print("\nAll containers stopped. Re-run Section 8 to start them again.")
else:
    print(f"\nStop exited with code {result.returncode}.")
    print("You can also stop manually: docker compose -p mdx stop")

## 14. Teardown

Stop all containers and **delete all data** (volumes, models, data directory). Run the cell below when you want to completely remove the deployment.

This runs `dev-profile.sh down` which stops containers, removes networks, and deletes the data directory. If Docker storage was moved to NVMe (Section 4), volume cleanup requires an extra step because Docker can't remove volumes whose data lives outside its data-root (the symlink trick). The cell handles this automatically.

In [ ]:
import subprocess, os, json

# --- Run dev-profile.sh down ---
print("Tearing down VSS deployment...")
process = subprocess.Popen(
    ["bash", os.path.join(SCRIPT_DIR, "dev-profile.sh"), "down"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=SCRIPT_DIR
)
for line in process.stdout:
    print(line, end="")
process.wait()
print(f"\nTeardown exit code: {process.returncode}")

# --- Clean up stuck volumes ---
# When Docker's data-root is on NVMe but volumes are symlinked back to root,
# `docker volume rm` fails with "unable to remove a directory outside of the
# local volume root". Fall back to sudo rm for those, then restart Docker.

result = subprocess.run(["docker", "volume", "ls", "-q"], capture_output=True, text=True)
leftover = result.stdout.strip().splitlines()

if leftover:
    print(f"\n{len(leftover)} leftover volume(s). Cleaning up...")
    need_restart = False
    for vol in leftover:
        r = subprocess.run(["docker", "volume", "rm", "-f", vol],
                          capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  removed  {vol}")
        else:
            # Symlinked volume — remove directly from /var/lib/docker/volumes
            vol_path = f"/var/lib/docker/volumes/{vol}"
            r2 = subprocess.run(["sudo", "rm", "-rf", vol_path],
                               capture_output=True, text=True)
            if r2.returncode == 0:
                print(f"  rm'd     {vol}")
                need_restart = True
            else:
                print(f"  FAILED   {vol}: {r2.stderr.strip()}")

    if need_restart:
        print("\n  Restarting Docker to clear volume metadata...")
        subprocess.run(["sudo", "systemctl", "restart", "docker"],
                      capture_output=True, check=True)

    # Verify
    result = subprocess.run(["docker", "volume", "ls", "-q"], capture_output=True, text=True)
    remaining = result.stdout.strip().splitlines()
    if remaining:
        print(f"\n  {len(remaining)} volume(s) still stuck:")
        for v in remaining:
            print(f"    {v}")
    else:
        print("\nAll volumes cleaned up.")
else:
    print("\nAll volumes cleaned up.")

In [ ]:
# To remove the deployment repo from disk:
# import shutil
# shutil.rmtree(REPO_DIR)
# print(f"Removed {REPO_DIR}")